In [3]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv

In [5]:
load_dotenv()
embedding_model = HuggingFaceEndpointEmbeddings(
    model = "sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
## create langchain documents for IPL players

# as the document object consists of page content and metadata discussed in document loaders
doc1 = Document(
    page_content="Virat Kohli is a right-handed batsman who plays for Royal Challengers Bengaluru. He is one of the most consistent run-scorers in IPL history with over 8000 runs.",
    metadata={"team": "rcb"}
)

doc2 = Document(
    page_content="MS Dhoni is a wicket-keeper batsman and the iconic captain of Chennai Super Kings. Known for his finishing ability and calm decision-making under pressure.",
    metadata={"team": "csk"}
)

doc3 = Document(
    page_content="Rohit Sharma is a right-handed opener and former captain of Mumbai Indians. He is the highest run-scorer in IPL history and known for his elegant stroke play.",
    metadata={"team": "mi"}
)

doc4 = Document(
    page_content="Jasprit Bumrah is a right-arm fast bowler for Mumbai Indians. Known for his unorthodox action and lethal yorkers, he is considered one of the best death bowlers in T20 cricket.",
    metadata={"team": "mi"}
)

doc5 = Document(
    page_content="Rishabh Pant is a left-handed wicket-keeper batsman who plays for Delhi Capitals. Known for his aggressive batting style and ability to turn matches single-handedly.",
    metadata={"team": "dc"}
)

docs = [doc1,doc2,doc3,doc4,doc5]

In [7]:
## creating the vector store using the object of Chroma
vector_store = Chroma(
    embedding_function=embedding_model, # what model we are going to use to generate embeddings
    persist_directory='chroma_db', # the docs which are going to be stored where are they gonna store, so in the chroma_db in root directory
    collection_name= 'sample' # name of the collection or tables in the rdbms
)

In [ ]:
# collection created but is still empty
print(vector_store._collection.count())

0


In [ ]:
## add documents to the vector store
vector_store.add_documents(docs)

# ef90d11b-afbf-4361-9e10-954f61e63ba9 these are unique id for each document in the collection

['ef90d11b-afbf-4361-9e10-954f61e63ba9',
 '94f348a2-5415-4500-83f7-9983ee7e71dd',
 '380ebdee-7bff-4722-a2b5-1b9c788421f5',
 '61695bed-cde4-4964-afb9-558f11617964',
 '3e202b26-5f5a-4310-88db-bbc11869d74b']

In [ ]:
# fetch embeddings , doccuments and metadata using the get method of the vector store created
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['ef90d11b-afbf-4361-9e10-954f61e63ba9',
  '94f348a2-5415-4500-83f7-9983ee7e71dd',
  '380ebdee-7bff-4722-a2b5-1b9c788421f5',
  '61695bed-cde4-4964-afb9-558f11617964',
  '3e202b26-5f5a-4310-88db-bbc11869d74b'],
 'embeddings': array([[ 0.01804495,  0.06454499, -0.03983108, ..., -0.02172047,
          0.01138513,  0.02469367],
        [-0.05855458,  0.04584749,  0.00777763, ...,  0.00379553,
         -0.03674259, -0.02326832],
        [-0.01328556, -0.00867057, -0.02831063, ...,  0.001113  ,
         -0.02987492,  0.01735785],
        [ 0.00930427, -0.01683619, -0.0511191 , ..., -0.07538816,
          0.00933866,  0.10285518],
        [ 0.04192998,  0.02434112,  0.02591072, ..., -0.02470657,
          0.00266851,  0.11002883]], shape=(5, 384)),
 'documents': ['Virat Kohli is a right-handed batsman who plays for Royal Challengers Bengaluru. He is one of the most consistent run-scorers in IPL history with over 8000 runs.',
  'MS Dhoni is a wicket-keeper batsman and the iconic captai

In [13]:
## search documents 

vector_store.similarity_search(
    query='who among these is a bowler',
    k = 2 # how many similar objects we need to display
)

[Document(id='61695bed-cde4-4964-afb9-558f11617964', metadata={'team': 'mi'}, page_content='Jasprit Bumrah is a right-arm fast bowler for Mumbai Indians. Known for his unorthodox action and lethal yorkers, he is considered one of the best death bowlers in T20 cricket.'),
 Document(id='380ebdee-7bff-4722-a2b5-1b9c788421f5', metadata={'team': 'mi'}, page_content='Rohit Sharma is a right-handed opener and former captain of Mumbai Indians. He is the highest run-scorer in IPL history and known for his elegant stroke play.')]

In [16]:
## search with similarity score

vector_store.similarity_search_with_score(
    query = 'Who is the best player among them', # chikoo bolte dhoti kholte
    k = 1
)


[(Document(id='ef90d11b-afbf-4361-9e10-954f61e63ba9', metadata={'team': 'rcb'}, page_content='Virat Kohli is a right-handed batsman who plays for Royal Challengers Bengaluru. He is one of the most consistent run-scorers in IPL history with over 8000 runs.'),
  1.294424295425415)]

In [18]:
# metadata filtering 

vector_store.similarity_search_with_score(
    query = '',
    filter={'team':'csk'}
)

[(Document(id='94f348a2-5415-4500-83f7-9983ee7e71dd', metadata={'team': 'csk'}, page_content='MS Dhoni is a wicket-keeper batsman and the iconic captain of Chennai Super Kings. Known for his finishing ability and calm decision-making under pressure.'),
  1.7472245693206787)]

In [19]:
## updating the existing documents
update_doc1 = Document(
    page_content='Virat Kohli the former captain of RCB is renowned for his aggressive leadership and consistent run scoring',
    metadata = {"team": "rcb"}
)

vector_store.update_document(document=update_doc1,document_id='ef90d11b-afbf-4361-9e10-954f61e63ba9')

In [22]:
vector_store.get(include = ['documents','embeddings','metadatas'])

{'ids': ['ef90d11b-afbf-4361-9e10-954f61e63ba9',
  '94f348a2-5415-4500-83f7-9983ee7e71dd',
  '380ebdee-7bff-4722-a2b5-1b9c788421f5',
  '61695bed-cde4-4964-afb9-558f11617964',
  '3e202b26-5f5a-4310-88db-bbc11869d74b'],
 'embeddings': array([[ 0.01497995,  0.07268309, -0.10414591, ..., -0.03880668,
          0.04681738, -0.04582775],
        [-0.05855458,  0.04584749,  0.00777763, ...,  0.00379553,
         -0.03674259, -0.02326832],
        [-0.01328556, -0.00867057, -0.02831063, ...,  0.001113  ,
         -0.02987492,  0.01735785],
        [ 0.00930427, -0.01683619, -0.0511191 , ..., -0.07538816,
          0.00933866,  0.10285518],
        [ 0.04192998,  0.02434112,  0.02591072, ..., -0.02470657,
          0.00266851,  0.11002883]], shape=(5, 384)),
 'documents': ['Virat Kohli the former captain of RCB is renowned for his aggressive leadership and consistent run scoring',
  'MS Dhoni is a wicket-keeper batsman and the iconic captain of Chennai Super Kings. Known for his finishing abili

In [23]:
## deleting the document 

vector_store.delete(ids=['3e202b26-5f5a-4310-88db-bbc11869d74b','61695bed-cde4-4964-afb9-558f11617964'])# we can delete by id 

In [24]:
vector_store.get(include = ['documents','embeddings','metadatas'])

{'ids': ['ef90d11b-afbf-4361-9e10-954f61e63ba9',
  '94f348a2-5415-4500-83f7-9983ee7e71dd',
  '380ebdee-7bff-4722-a2b5-1b9c788421f5'],
 'embeddings': array([[ 0.01497995,  0.07268309, -0.10414591, ..., -0.03880668,
          0.04681738, -0.04582775],
        [-0.05855458,  0.04584749,  0.00777763, ...,  0.00379553,
         -0.03674259, -0.02326832],
        [-0.01328556, -0.00867057, -0.02831063, ...,  0.001113  ,
         -0.02987492,  0.01735785]], shape=(3, 384)),
 'documents': ['Virat Kohli the former captain of RCB is renowned for his aggressive leadership and consistent run scoring',
  'MS Dhoni is a wicket-keeper batsman and the iconic captain of Chennai Super Kings. Known for his finishing ability and calm decision-making under pressure.',
  'Rohit Sharma is a right-handed opener and former captain of Mumbai Indians. He is the highest run-scorer in IPL history and known for his elegant stroke play.'],
 'uris': None,
 'included': ['documents', 'embeddings', 'metadatas'],
 'data'